# DeepTrack2 dogfooding: does KíLín answer questions about a third-party codebase correctly?

**The Problem.** Can KíLín's real end-to-end chat pipeline honors its own system prompt's contract 
when asked about a codebase KíLín had no part in writing?

**The Data.** [DeepTrack2](https://github.com/DeepTrackAI/DeepTrack2) is
a modular Python library for image data pipelines used in microscopy. DeepTrack2 is close in scale to KíLín's own package -- the right
size for a hand-authored, human-verified gold set rather than something needing dozens of questions to
say anything meaningful.  The corpus covers the `deeptrack/` package *and* the `tutorials/` notebooks, not the package alone.
DeepTrack2 ships an extensive `tutorials/` tree (getting-started, real-world examples, advanced topics,
developer guides -- all `.ipynb`) alongside the `deeptrack/` package itself, and a lot of the real,
intended-usage knowledge (how a module is actually meant to be used, not just its docstring) lives
there rather than in the package's own source.

**The Metrics.** Two tiers. Tier 1 is fully mechanical and runs automatically in this
notebook: citation validity (every `[n]` in an answer refers to evidence KíLín actually supplied),
citation coverage (of a question's gold-relevant evidence, how much actually got cited), abstention
correctness on the negative questions, and verbatim-copy detection (the system prompt forbids
copying source text directly). Tier 2 -- does each cited claim actually hold up, does the answer
satisfy its fact checklist -- needs judgment a regex can't provide, and this notebook exports it as a
worksheet for you to grade by hand.

**Reproducibility.** The default run uses a fixed search intent (`USE_ROUTER=False`). Set it to True to assess KíLín intent routing, but such runs are not directly comparable. DeepTrack2 gold citations are preserved as provided and require validation against the selected external DeepTrack checkout.

## 1. Prepare KíLín

Locate the KíLín repo root, put it on the import path, and run KíLín's notebook setup.

In [ ]:
from pathlib import Path

root = Path.cwd()
while not (root / "kilin").is_dir() and root != root.parent:
    root = root.parent

if not (root / "kilin").is_dir():
    raise RuntimeError("Open the notebook from the KíLín checkout.")

# The benchmark helpers and run folders live in _si_article_/benchmarks/,
# imported as the benchmarks package from its parent folder.
import sys

benchmarks_dir = root / "_si_article_" / "benchmarks"
if str(benchmarks_dir.parent) not in sys.path:
    sys.path.insert(0, str(benchmarks_dir.parent))

print(f"KíLín root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

## 2. Point at a local DeepTrack2 checkout

Set `DEEPTRACK_SOURCE` below to wherever your own checkout lives. The default guess assumes it sits beside this repo (`../DeepTrack2`, the common layout when both are cloned under the same GitHub folder); edit it if yours is somewhere else. This should be the repo root (containing both `deeptrack/` and `tutorials/`), not the 
`deeptrack/` package subfolder.

In [ ]:
DEEPTRACK_SOURCE = (
    root.parent / "DeepTrack2"
)  # <-- edit if your checkout lives elsewhere

if not DEEPTRACK_SOURCE.is_dir():
    raise FileNotFoundError(
        f"No DeepTrack2 checkout found at {DEEPTRACK_SOURCE}. "
        "Set DEEPTRACK_SOURCE above to your local clone of "
        "DeepTrackAI/DeepTrack2."
    )

deeptrack_package = DEEPTRACK_SOURCE / "deeptrack"
if not deeptrack_package.is_dir():
    raise FileNotFoundError(
        f"{DEEPTRACK_SOURCE} doesn't contain a deeptrack/ package -- "
        "check DEEPTRACK_SOURCE points at the repo root, not a subfolder."
    )

tutorials_dir = DEEPTRACK_SOURCE / "tutorials"
if not tutorials_dir.is_dir():
    print(
        f"NOTE: no tutorials/ folder found at {tutorials_dir} "
        "-- indexing deeptrack/ only."
    )

print(f"DeepTrack2 checkout: {DEEPTRACK_SOURCE}")
print(f"Package: {deeptrack_package}  (exists)")
print(
    f"Tutorials: {tutorials_dir}  "
    f"({'exists' if tutorials_dir.is_dir() else 'missing'})"
)

## 3. Load the gold question set

The questions and their gold citations are hand-authored and checked
into the repo ()`benchmarks/gold/dogfood_deeptrack_code.json`). 
`load_generation_tasks` (`benchmarks/common.py`)
turns each entry into a `GenerationTask`.

In [ ]:
from benchmarks.common import load_generation_tasks

gold_path = benchmarks_dir / "gold" / "dogfood_deeptrack_code.json"
tasks = load_generation_tasks(gold_path)

abstention_count = sum(1 for task in tasks if task.requires_abstention)
print(
    f"Loaded {len(tasks)} questions "
    f"({abstention_count} deliberate abstention questions)."
)
for task in tasks:
    marker = " [abstention]" if task.requires_abstention else ""
    print(f"  {task.task_id}{marker}: {task.question}")

if not tasks:
    print(
        "\nGold set is empty -- add tasks to",
        gold_path.relative_to(root),
        "before this is a real benchmark.",
    )

## 4. Open DeepTrack2 as a real KíLín project

`prepare_project_workspace` points at the whole `DEEPTRACK_SOURCE`
repo root so both `deeptrack/` and `tutorials/` get mirrored. Mirrors it into a
managed workspace under `benchmarks/sandbox/`, tracked with a SHA-256 manifest -- reopening after a
local DeepTrack2 change picks up only what actually changed.

`.ipynb` is on by default for a "code" kind project. One setting still needs adjusting from the default, applied via KíLín's own `configure_project` (not a raw file edit): the default 1 MB per-file limit is sized for source
files, not notebooks with embedded plot outputs.

In [ ]:
from kilin.project import (
    configure_project,
    prepare_project_workspace,
    synchronize_project_workspace,
)

workspace_root = benchmarks_dir / "sandbox"
project = prepare_project_workspace(
    source_directory=DEEPTRACK_SOURCE,
    type_id="code",
    workspace_root=workspace_root,
)

# Allow large example notebooks without widening core defaults.
DESIRED_MAX_FILE_BYTES = 5 * 1024 * 1024

if project.profile.max_file_bytes != DESIRED_MAX_FILE_BYTES:
    project = configure_project(
        project.root,
        {"max_file_bytes": DESIRED_MAX_FILE_BYTES},
    )
    # Setting changed after the mirror above was built under the old limit --
    # Synchronize the SAME configured workspace. A new prepare
    # call allocates another project with the old defaults.
    project = synchronize_project_workspace(
        DEEPTRACK_SOURCE, project.root
    )

# Use the core-owned starting models instead of duplicating model IDs.
# The recorded IDs make run-to-run model changes visible.
from kilin.model_defaults import (
    DEFAULT_CHAT_MODEL_MLX,
    DEFAULT_CHAT_MODEL_TORCH,
    DEFAULT_EMBEDDING_MODEL_MLX,
    DEFAULT_EMBEDDING_MODEL_TORCH,
)

DESIRED_MODEL_SETTINGS = {
    "chat_mlx": DEFAULT_CHAT_MODEL_MLX,
    "chat_torch": DEFAULT_CHAT_MODEL_TORCH,
    "embedding_mlx": DEFAULT_EMBEDDING_MODEL_MLX,
    "embedding_torch": DEFAULT_EMBEDDING_MODEL_TORCH,
}
if any(
    getattr(project.profile, key) != value
    for key, value in DESIRED_MODEL_SETTINGS.items()
):
    project = configure_project(project.root, DESIRED_MODEL_SETTINGS)
    print(
        "Model settings corrected to this benchmark's known-good "
        "configuration."
    )

print(f"Project root: {project.root.relative_to(root)}")
print(f"Project source: {project.source}")
print(f"Extensions indexed: {project.profile.extensions}")
print(f"Max file size: {project.profile.max_file_bytes:,} bytes")
print(f"Chat model (mlx): {project.profile.chat_mlx}")
print(f"Embedding model (mlx): {project.profile.embedding_mlx}")
print(
    f"Chunking: target={project.profile.target_tokens} "
    f"max={project.profile.max_tokens} "
    f"overlap={project.profile.overlap_tokens}"
)

## 5. Open a real RAGSession

Loads both models and builds (or reuses) the semantic, lexical, and code-graph indexes over DeepTrack2.
`progress_callback` prints KíLín's own structured progress events so a first run (no cached index yet)
doesn't sit silently .

In [ ]:
from kilin.rag import RAGSession


def _print_progress(event):
    if event.stage in {"ready"} or event.current is None:
        print(f"[{event.percent:5.1f}%] {event.message}")


rag_session = RAGSession(project.root, progress_callback=_print_progress)

print(f"Backend: {rag_session.backend}")
print(f"Index: {rag_session.index_status}")
print(f"Chunks: {len(rag_session.chunks)}")

## 6. Configure answer guidance

Set a fixed optional answer instruction without modifying private chat
state. KíLín composes its effective system prompt from project-type
configuration on each turn; assigning to `_system_prompt` would not
provide a reliable benchmark override. Disabling the router keeps the
experiment focused on evidence retrieval, citations, and generation.


In [ ]:
USE_STRONGER_GUIDANCE = True
USE_ROUTER = False

# Keep the fixed evidence-search intent for citation benchmarks. A
# routed experiment can be run by setting USE_ROUTER to True, but
# tool-generated answers then measure a different retrieval pipeline.
_STRONGER_ANSWER_GUIDANCE = (
    "Answer only from the supplied project evidence. Cite each "
    "supported claim using [1], [2], and so on. If the evidence "
    "clearly does not answer the question, respond with exactly: "
    "Insufficient evidence. Do not guess or answer from general "
    "knowledge."
)

ANSWER_GUIDANCE = (
    _STRONGER_ANSWER_GUIDANCE if USE_STRONGER_GUIDANCE else None
)
print(f"Router enabled: {USE_ROUTER}")
print(f"Additional answer guidance: {ANSWER_GUIDANCE!r}")

## 7. Check gold citations against the real, chunked corpus

Confirms each question's gold citation(s) actually land inside a real chunk before asking anything --
catches a stale line number (e.g. after a DeepTrack2 update) before every question has already run.

In [ ]:
from benchmarks.common import gold_evidence_indices


class _ChunkAsEvidence:
    """Adapt a real Chunk to the same (path, start_line, end_line, cell,
    citation) shape gold_evidence_indices expects from an Evidence --
    citation is unused for this check, only path/cell/line overlap matters.
    cell matters for a notebook-derived chunk (tutorials/*.ipynb): every
    cell's line numbers start at 1, so a gold citation naming a specific
    cell ("path#cell:1-8") needs the real cell number to disambiguate, not
    just the line range.
    """

    __slots__ = ("citation", "path", "start_line", "end_line", "cell")

    def __init__(self, chunk):
        self.citation = 0
        self.path = chunk.path
        self.start_line = chunk.start_line
        self.end_line = chunk.end_line
        self.cell = chunk.cell


chunk_evidence = [_ChunkAsEvidence(chunk) for chunk in rag_session.chunks]

unmatched = []
for task in tasks:
    if task.requires_abstention:
        continue
    if not gold_evidence_indices(chunk_evidence, task.gold_citations):
        unmatched.append(task.task_id)

if unmatched:
    print(
        f"WARNING: {len(unmatched)} question(s) have a gold citation that "
        f"matches no real chunk: {unmatched}"
    )
    print(
        "Check these against the current deeptrack/ or tutorials/ source "
        "before trusting their scores below."
    )
else:
    answerable = len(tasks) - sum(1 for t in tasks if t.requires_abstention)
    print(
        f"All {answerable} answerable questions' gold citations match a real "
        "chunk."
    )

## 8. Ask every question for real

`rag_session.reset()` before each question keeps every turn an independent, single-turn conversation.
`rag_session.turns[-1].evidence` after `ask()` is the exact, real `Evidence` KíLín retrieved and
supplied for that answer.

In [ ]:
from tqdm.auto import tqdm

run_results: list[dict[str, object]] = []

for task in tqdm(tasks, desc="Asking", unit="question"):
    rag_session.reset()
    answer = rag_session.ask(
        task.question,
        use_router=USE_ROUTER,
        answer_guidance=ANSWER_GUIDANCE,
    )
    turn = rag_session.turns[-1]
    run_results.append(
        {
            "task_id": task.task_id,
            "question": task.question,
            "answer": answer,
            "evidence": [
                {
                    "citation": item.citation,
                    "path": str(item.path),
                    "start_line": item.start_line,
                    "end_line": item.end_line,
                    "symbol": item.symbol,
                    "text": item.text,
                    "cell": item.cell,
                }
                for item in turn.evidence
            ],
        }
    )

print(f"Asked {len(run_results)} questions.")

## 9. Score Tier 1 (mechanical)

Same mechanics as notebook 60: citation validity, citation coverage, abstention correctness, and
verbatim-copy detection.

In [ ]:
from dataclasses import dataclass as _dataclass
from pathlib import Path as _Path

from benchmarks.common import (
    abstains,
    citation_coverage,
    citation_validity,
    verbatim_overlap_chars,
)

# Long verbatim overlaps indicate possible excessive copying.
VERBATIM_CHAR_THRESHOLD = 60


@_dataclass
class _EvidenceView:
    citation: int
    path: _Path
    start_line: int
    end_line: int
    symbol: object
    text: str
    cell: int | None = None


tasks_by_id = {task.task_id: task for task in tasks}
tier1_rows: list[dict[str, object]] = []

for row in run_results:
    task = tasks_by_id[row["task_id"]]
    evidence = [_EvidenceView(**item) for item in row["evidence"]]

    valid, invalid_indices = citation_validity(
        row["answer"], evidence_count=len(evidence)
    )
    coverage = (
        None
        if task.requires_abstention
        else citation_coverage(row["answer"], evidence, task.gold_citations)
    )
    abstained = abstains(row["answer"])
    verbatim_chars = verbatim_overlap_chars(row["answer"], evidence)

    tier1_rows.append(
        {
            "task_id": task.task_id,
            "requires_abstention": task.requires_abstention,
            "citation_valid": valid,
            "invalid_citation_indices": invalid_indices,
            "citation_coverage": coverage,
            "abstained": abstained,
            "abstention_correct": (
                (abstained if task.requires_abstention else not abstained)
            ),
            "verbatim_overlap_chars": verbatim_chars,
            "verbatim_flag": (
                verbatim_chars is not None
                and verbatim_chars >= VERBATIM_CHAR_THRESHOLD
            ),
        }
    )

valid_count = sum(1 for r in tier1_rows if r["citation_valid"])
abstention_correct_count = sum(
    1 for r in tier1_rows if r["abstention_correct"]
)
coverage_values = [
    r["citation_coverage"]
    for r in tier1_rows
    if r["citation_coverage"] is not None
]
verbatim_flags = [r["task_id"] for r in tier1_rows if r["verbatim_flag"]]

print(
    f"Citation validity: {valid_count}/{len(tier1_rows)} answers cite only "
    "real evidence indices."
)
print(
    f"Abstention correctness: {abstention_correct_count}/{len(tier1_rows)} "
    f"({abstention_count} of these are the deliberate negatives)."
)
if coverage_values:
    print(
        f"Mean citation coverage (answerable questions with retrieved gold "
        f"evidence, n={len(coverage_values)}): "
        f"{sum(coverage_values) / len(coverage_values):.2f}"
    )
else:
    print(
        "No answerable question had its gold evidence retrieved "
        "-- coverage undefined."
    )
print(
    f"Verbatim-copy flags (>= {VERBATIM_CHAR_THRESHOLD} chars): "
    f"{verbatim_flags or 'none'}"
)

## 10. Export the Tier 2 grading worksheet

Same as notebook 60: faithfulness and fact-checklist satisfaction need a human reading the answer, not
a regex -- no LLM-judge API call wired in here, by design.

In [ ]:
worksheet_lines = []
for row, tier1 in zip(run_results, tier1_rows):
    task = tasks_by_id[row["task_id"]]
    worksheet_lines.append(
        f"## {task.task_id}"
        f"{' [abstention]' if task.requires_abstention else ''}"
    )
    worksheet_lines.append(f"**Question:** {task.question}")
    worksheet_lines.append("")
    worksheet_lines.append(f"**Answer:**\n{row['answer']}")
    worksheet_lines.append("")
    worksheet_lines.append("**Evidence supplied:**")
    for item in row["evidence"]:
        symbol = f" ({item['symbol']})" if item["symbol"] else ""
        worksheet_lines.append(
            f"- [{item['citation']}] {item['path']}:"
            f"{item['start_line']}-{item['end_line']}{symbol}"
        )
    worksheet_lines.append("")
    worksheet_lines.append("**Fact checklist:**")
    for fact in task.fact_checklist:
        worksheet_lines.append(f"- [ ] {fact}")
    worksheet_lines.append("")
    if task.reference_answer:
        worksheet_lines.append(
            "**Reference reply (from the fact checklist -- a comparison "
            "aid, not ground truth on its own):**"
        )
        worksheet_lines.append(task.reference_answer)
        worksheet_lines.append("")
    worksheet_lines.append(
        f"**Tier 1 (automatic):** citation_valid={tier1['citation_valid']}, "
        f"citation_coverage={tier1['citation_coverage']}, "
        f"abstained={tier1['abstained']}, "
        f"abstention_correct={tier1['abstention_correct']}, "
        f"verbatim_overlap_chars={tier1['verbatim_overlap_chars']}"
    )
    worksheet_lines.append("")
    worksheet_lines.append(
        "**Tier 2 (your read):** faithful? [ ]  checklist satisfied? [ ]  "
        "notes: "
    )
    worksheet_lines.append("\n---\n")

worksheet_text = "\n".join(worksheet_lines)
worksheet_path = benchmarks_dir / "results"
worksheet_path.mkdir(parents=True, exist_ok=True)

## 11. Save results

Same pattern as notebook 60, distinguished by filename (`dogfood_deeptrack_code` instead of
`dogfood_code`) so results from the two corpora never collide in `benchmarks/results/`.

In [ ]:
import json as _json
from benchmarks.common import local_run_timestamp

results_dir = benchmarks_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp, stamp = local_run_timestamp()

run_record = {
    "benchmark": "KíLín dogfooding -- DeepTrack2, "
    "hand-authored gold questions (Area A: code)",
    "timestamp": timestamp,
    "corpus_source": str(DEEPTRACK_SOURCE),
    "chat_model": project.profile.chat_model(rag_session.backend),
    "embedding_model": project.profile.embedding_model(rag_session.backend),
    "backend": rag_session.backend,
    "index_status": rag_session.index_status,
    "chunk_count": len(rag_session.chunks),
    "question_count": len(tasks),
    "abstention_question_count": sum(
        1 for t in tasks if t.requires_abstention
    ),
    "gold_citation_unmatched": unmatched,
    "answer_guidance": ANSWER_GUIDANCE,
    "use_router": USE_ROUTER,
    "tier1_summary": {
        "citation_valid_count": valid_count,
        "abstention_correct_count": abstention_correct_count,
        "mean_citation_coverage": (
            sum(coverage_values) / len(coverage_values)
            if coverage_values
            else None
        ),
        "verbatim_flagged_task_ids": verbatim_flags,
    },
    "per_question": [
        {**row, "tier1": tier1} for row, tier1 in zip(run_results, tier1_rows)
    ],
}

output_path = results_dir / f"{stamp}_dogfood_deeptrack_code.json"
output_path.write_text(_json.dumps(run_record, indent=2))

worksheet_output_path = (
    results_dir / f"{stamp}_dogfood_deeptrack_code_tier2_worksheet.md"
)
worksheet_output_path.write_text(worksheet_text)

print(f"Saved: {output_path.relative_to(root)}")
print(f"Saved: {worksheet_output_path.relative_to(root)}")

## 12. Release the models

In [ ]:
rag_session.close()

print("Chat and embedding models released.")